# Gemini 3 Batch Transcription Pipeline

This Colab orchestrates the transcription of audio segments using Google's **Gemini 3** model.

*Note:* This Colab uses the output creating using the `chirp_1_segment_label_studio_examples.ipynb` notebook.

### Workflow Overview:
1.  **Job Submission**: Reads a JSONL manifest from GCS and submits asynchronous batch jobs in chunks of 15 segments. It automatically filters out files without labels to optimize API costs.
2.  **Organization**: Results are written to GCS (`gs://wd-asr-chirp-evaluation/gemini_transcripts`).

In [1]:
import sys
!pip install loguru

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.6/61.6 kB 1.2 MB/s eta 0:00:00


In [2]:
from google.colab import auth

from google import genai
from google.genai import types
from google.cloud import storage

from loguru import logger

import json
import os
import time
import sys
import re
import asyncio

from tenacity import retry, stop_after_attempt, wait_exponential, retry_if_exception_type

from tqdm.auto import tqdm

In [3]:
# @title Define constants and initial logging
MODEL_ID = "gemini-3.1-pro-preview" # Need to also evaluate the Flash model!
RESPONSE_SCHEMA = {
    "type": "object",
    "properties": {
        "transcription": {"type": "string"}
    },
    "required": ["transcription"]
}

GCP_PROJECT_ID = "automatic-hawk-481415-m9"
GCS_BUCKET="wd-asr-chirp-evaluation"
GCS_INPUT_DIR="segmented_audio"
MODEL_ID_DIR = re.sub(r'[-\.]', '_', MODEL_ID)
GCS_OUTPUT_DIR=f"{MODEL_ID_DIR}_transcripts"
GCP_LOCATION="global"

# Segmentation manifest path
MANIFEST_URI = f"gs://{GCS_BUCKET}/{GCS_INPUT_DIR}/batch_manifest.jsonl"

SYSTEM_PROMPT = """
You are an expert transcriptionist specializing in emergency dispatch radio communications.

Your goal is to provide the most accurate verbatim transcript possible of the provided audio.

STRICT RULES:
1. ONLY provide the transcription text with no newlines.
2. When transcribing numbers, write the digits (e.g., write 1.7 instead of 'one point seven', and 100 instead of 'one hundred').
3. Use standard punctuation and capitalization to make the transcript readable.
4. Do NOT attempt to guess unintelligible words. If a word is not clearly identifiable, simply omit it.
5. Do not add metadata, annotations, or descriptions of background noise.
6. Maintain professional terminology and codes exactly as spoken.
"""

# Initialize loguru
logger.remove()
logger.add(sys.stderr, format="<level>{level}</level>: {message}");

In [4]:
# @title Authenticate with GCP
auth.authenticate_user()

!gcloud config set project {GCP_PROJECT_ID} --quiet

Updated property [core/project].


To take a quick anonymous survey, run:
  $ gcloud survey



In [5]:
# @title Allow access to the GCS bucket to the VertexAI Service Agent
%%bash
PROJECT_ID="automatic-hawk-481415-m9"
PROJECT_NUM=$(gcloud projects describe $PROJECT_ID --format='value(projectNumber)')
AGENT="service-$PROJECT_NUM@gcp-sa-aiplatform.iam.gserviceaccount.com"

gcloud storage buckets add-iam-policy-binding gs://wd-asr-chirp-evaluation \
    --member="serviceAccount:$AGENT" \
    --role="roles/storage.objectViewer"

bindings:
- members:
  - projectEditor:automatic-hawk-481415-m9
  - projectOwner:automatic-hawk-481415-m9
  role: roles/storage.legacyBucketOwner
- members:
  - projectViewer:automatic-hawk-481415-m9
  role: roles/storage.legacyBucketReader
- members:
  - projectEditor:automatic-hawk-481415-m9
  - projectOwner:automatic-hawk-481415-m9
  role: roles/storage.legacyObjectOwner
- members:
  - projectViewer:automatic-hawk-481415-m9
  role: roles/storage.legacyObjectReader
- members:
  - serviceAccount:service-781667204380@gcp-sa-speech.iam.gserviceaccount.com
  role: roles/storage.objectAdmin
- members:
  - serviceAccount:service-781667204380@gcp-sa-aiplatform.iam.gserviceaccount.com
  role: roles/storage.objectViewer
etag: CAY=
kind: storage#policy
resourceId: projects/_/buckets/wd-asr-chirp-evaluation
version: 1


In [6]:
# Retry on 503 (Unavailable), 504 (Timeout), 500 (Internal Error), 429 (Resource Exhausted)
def is_transient_error(exception):
    err_msg = str(exception)
    return "500" in err_msg or "503" in err_msg or "504" in err_msg or "429" in err_msg

@retry(
    retry=retry_if_exception_type(Exception),
    stop=stop_after_attempt(3),
    wait=wait_exponential(multiplier=1, min=2, max=10),
    before_sleep=lambda retry_state: logger.warning(f"Retrying after error: {retry_state.outcome.exception()}")
)
async def process_audio_segment(entry, semaphore, genai_client, storage_client, output_base_path):
    uri = entry["audio_filepath"]
    fname_base = os.path.basename(uri).replace(".flac", "")

    async with semaphore:
        try:
            audio_part = types.Part.from_uri(
                file_uri=uri,
                mime_type="audio/flac"
            )

            response = await genai_client.aio.models.generate_content(
                model=MODEL_ID,
                contents=[audio_part, "Transcribe this audio verbatim."],
                config=types.GenerateContentConfig(
                    system_instruction=SYSTEM_PROMPT,
                    temperature=0.0,
                    response_mime_type="application/json",
                    response_schema=RESPONSE_SCHEMA,
                    thinking_config=types.ThinkingConfig(
                        thinking_level="LOW"
                    ),
                    safety_settings=[
                        types.SafetySetting(category="HARM_CATEGORY_HATE_SPEECH", threshold="BLOCK_NONE"),
                        types.SafetySetting(category="HARM_CATEGORY_HARASSMENT", threshold="BLOCK_NONE"),
                        types.SafetySetting(category="HARM_CATEGORY_DANGEROUS_CONTENT", threshold="BLOCK_NONE"),
                        types.SafetySetting(category="HARM_CATEGORY_SEXUALLY_EXPLICIT", threshold="BLOCK_NONE"),
                    ]
                )
            )

            # 1. Parse response.text using json.loads()
            try:
                res_data = json.loads(response.text)
            except json.JSONDecodeError:
                logger.error(f"Invalid JSON response for {uri}")
                return False

            # 2. Validation check for 'transcription' key
            transcription = res_data.get("transcription")
            if not isinstance(transcription, str) or not transcription.strip():
                logger.error(f"Validation failed: Missing or empty transcription for {uri}")
                return False

            output_path = f"{output_base_path}/{fname_base}.json"
            out_blob = storage_client.bucket(GCS_BUCKET).blob(output_path)

            # Upload to GCS
            await asyncio.to_thread(out_blob.upload_from_string, response.text, "application/json")

            # 3 & 4. Verification and return logic
            exists = await asyncio.to_thread(out_blob.exists)
            if exists:
                return True
            else:
                logger.error(f"GCS upload verification failed for {output_path}")
                return False

        except Exception as e:
            logger.error(f"Error processing {uri}: {e}")
            return False

async def execute_pipeline(overwrite=False):
    # Initialize clients
    storage_client = storage.Client(project=GCP_PROJECT_ID)
    genai_client = genai.Client(
        vertexai=True,
        project=GCP_PROJECT_ID,
        location=GCP_LOCATION
    )
    bucket = storage_client.bucket(GCS_BUCKET)

    processed_files = set()
    if overwrite:
        logger.info(f"Overwrite enabled. Clearing existing JSONs in {GCS_OUTPUT_DIR}...")
        blobs_to_delete = list(bucket.list_blobs(prefix=GCS_OUTPUT_DIR))
        for blob in blobs_to_delete:
            if blob.name.endswith('.json'):
                blob.delete()
    else:
        logger.info(f"Checking for existing transcripts in {GCS_OUTPUT_DIR}...")
        blobs = bucket.list_blobs(prefix=GCS_OUTPUT_DIR)
        processed_files = {os.path.basename(blob.name).replace('.json', '') for blob in blobs if blob.name.endswith('.json')}

    bucket_name = MANIFEST_URI.replace("gs://", "").split("/")[0]
    blob_path = "/".join(MANIFEST_URI.replace("gs://", "").split("/")[1:])
    manifest_content = storage_client.bucket(bucket_name).blob(blob_path).download_as_text()
    manifest_entries = [json.loads(line) for line in manifest_content.strip().split("\n") if line.strip()]

    # Filter manifest entries for delta-processing
    filtered_entries = []
    for entry in manifest_entries:
        uri = entry["audio_filepath"]
        fname_base = os.path.basename(uri).replace(".flac", "")

        if overwrite or fname_base not in processed_files:
            filtered_entries.append(entry)

    num_skipped = len(manifest_entries) - len(filtered_entries)
    logger.info(f"Total entries: {len(manifest_entries)} | Skipped: {num_skipped} | To process: {len(filtered_entries)}")

    if filtered_entries:
        # Limit concurrency to avoid exceeding service limits
        semaphore = asyncio.Semaphore(5)
        tasks = [
            process_audio_segment(entry, semaphore, genai_client, storage_client, GCS_OUTPUT_DIR)
            for entry in filtered_entries
        ]
        await tqdm.gather(*tasks, desc="Transcripts processed")
    else:
        logger.info("No new files to transcribe.")

    # --- Post-execution Integrity Check ---
    logger.info("Starting post-execution integrity check...")
    final_blobs = list(bucket.list_blobs(prefix=GCS_OUTPUT_DIR))
    final_processed_files = {os.path.basename(blob.name).replace('.json', '') for blob in final_blobs if blob.name.endswith('.json')}

    missing_files = []
    for entry in manifest_entries:
        uri = entry["audio_filepath"]
        fname_base = os.path.basename(uri).replace(".flac", "")
        if fname_base not in final_processed_files:
            missing_files.append(fname_base)

    if missing_files:
        logger.error(f"Integrity check FAILED. {len(missing_files)} files missing or invalid: {missing_files}")
    else:
        logger.info("Integrity check PASSED. All manifest entries accounted for.")

    logger.info(f"--- Execution Summary ---")
    logger.info(f"Total Manifest Entries: {len(manifest_entries)}")
    logger.info(f"Valid JSONs in GCS: {len(final_processed_files)}")
    logger.info(f"Output directory: gs://{GCS_BUCKET}/{GCS_OUTPUT_DIR}")

In [7]:
# @title Perform transcriptions
await execute_pipeline(overwrite=False)

INFO: Checking for existing transcripts in gemini_3_1_pro_preview_transcripts...
INFO: Total entries: 35 | Skipped: 35 | To process: 0
INFO: No new files to transcribe.
INFO: Starting post-execution integrity check...
INFO: Integrity check PASSED. All manifest entries accounted for.
INFO: --- Execution Summary ---
INFO: Total Manifest Entries: 35
INFO: Valid JSONs in GCS: 35
INFO: Output directory: gs://wd-asr-chirp-evaluation/gemini_3_1_pro_preview_transcripts
